In [6]:
import os 
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
import scvi
import anndata
import numpy as np
import pandas as pd 
import scanpy as sc 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [3]:
import anndata 
adata = anndata.read_h5ad(r'/mnt/xuisebnas2/Lab/Bereket_SEA_AD_analysis/combined_anndata_object/SEAAD_MTG_MERFISH.2024-12-11.h5ad')
adata

AnnData object with n_obs × n_vars = 1887729 × 180
    obs: 'Donor ID', 'Sex', 'Gender', 'Age at Death', 'Race (choice=White)', 'Race (choice=Black/ African American)', 'Race (choice=Asian)', 'Race (choice=American Indian/ Alaska Native)', 'Race (choice=Native Hawaiian or Pacific Islander)', 'Race (choice=Unknown or unreported)', 'Race (choice=Other)', 'specify other race', 'Hispanic/Latino', 'Highest level of education', 'Years of education', 'PMI', 'Fresh Brain Weight', 'Brain pH', 'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score', 'Overall CAA Score', 'Highest Lewy Body Disease', 'Total Microinfarcts (not observed grossly)', 'Total microinfarcts in screening sections', 'Atherosclerosis', 'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Last CASI Score', 'Interval from last CASI in months', 'Last MMSE Score', 'Interval from last MMSE in months', 'Last MOCA Score', 'Interval from last MOCA in months', 'APOE4 Status', 'Primary Study Name', 'Secondary Study Name', 

In [4]:
adata.obs.columns

Index(['Donor ID', 'Sex', 'Gender', 'Age at Death', 'Race (choice=White)',
       'Race (choice=Black/ African American)', 'Race (choice=Asian)',
       'Race (choice=American Indian/ Alaska Native)',
       'Race (choice=Native Hawaiian or Pacific Islander)',
       'Race (choice=Unknown or unreported)', 'Race (choice=Other)',
       'specify other race', 'Hispanic/Latino', 'Highest level of education',
       'Years of education', 'PMI', 'Fresh Brain Weight', 'Brain pH',
       'Overall AD neuropathological Change', 'Thal', 'Braak', 'CERAD score',
       'Overall CAA Score', 'Highest Lewy Body Disease',
       'Total Microinfarcts (not observed grossly)',
       'Total microinfarcts in screening sections', 'Atherosclerosis',
       'Arteriolosclerosis', 'LATE', 'Cognitive Status', 'Last CASI Score',
       'Interval from last CASI in months', 'Last MMSE Score',
       'Interval from last MMSE in months', 'Last MOCA Score',
       'Interval from last MOCA in months', 'APOE4 Status',
 

In [5]:
adata.obs[['LATE','Cognitive Status','Continuous Pseudo-progression Score']]

,LATE,Cognitive Status,Continuous Pseudo-progression Score
145223,LATE Stage 2,No dementia,0.204485
1018941,Not Identified,No dementia,0.182370
458233,LATE Stage 1,Dementia,0.725006
144812,LATE Stage 2,No dementia,0.204485
1419315,LATE Stage 1,No dementia,0.296555
...,...,...,...
1494334,LATE Stage 2,No dementia,0.522889
1075942,LATE Stage 1,No dementia,0.355632
1480879,LATE Stage 2,No dementia,0.522889
1294837,LATE Stage 2,No dementia,0.522889


In [7]:
import matplotlib.pyplot as plt
import seaborn as sns 
def plot_cluster_scdata(scdata,cmap,all_clusters,clusters=[1,2],bg_color = 'gray',bg_pt_size = 5, point_sz_map = None, transpose=1,flipx=1,flipy=1,hue_input = 'leiden' , ax = None,use_seaborn = False,show_title = True):
    """
    plot spatial location of cell cluster using either seaborn or matplotlib
    use seaborn for individual clusters
    use matplotlib scatter function to visualize all clusters
    cmap - dictionary that has color of each cluster  used in umap
    """
    fig = None
    if not ax:
     fig,ax = plt.subplots(figsize=(15, 15),)# facecolor="black")

    
    if use_seaborn:
        x,y,hue = 'x', 'y',hue_input
        sns.scatterplot(x=x, y=y, data=scdata.obs, 
                        s=bg_pt_size, markers=dict(marker_edge_color='none'),
                        rasterized=True,
                        ax=ax,c= bg_color)
    else: # use matplotlib
        x,y = (np.array(scdata.obs[['x', 'y']])*[flipx,flipy])[:,::transpose].T
        ax.scatter(x, y, c=bg_color, s = bg_pt_size, marker='.')
        
    for cluster in clusters:
        clst_idx = all_clusters.index(cluster)
        cluster_ = str(cluster)
        inds = scdata.obs[hue_input] == cluster_
        
        #col = cmap[int(clst_idx) % len(cmap)]
        col = cmap[cluster_] # cmap is dictionary
        point_sz = point_sz_map[cluster_] # point size dictionary
        if use_seaborn:
            sns.scatterplot(x=x, y=y, data=scdata.obs[inds], 
                        s=point_sz, markers=dict(marker_edge_color=col),
                        rasterized=True,
                        ax=ax,c= col)
            if show_title:
                ax.set_title(f'cluster {cluster_}',fontsize = 7, y= 1)
        else:
            x_ = x[inds]
            y_ = y[inds]
            ax.scatter(x_, y_, c=col, s= point_sz, marker='.',label = cluster_)
        
    # ax.set_xlim([x.min(),x.max()])
    # ax.set_ylim([y.min(),y.max()])
    ax.grid(False)
    ax.axis("off")
    ax.axis("equal")
    if fig:
        ax.legend( loc = 'lower center', ncol = len(clusters)//3,markerscale = 12, title = 'Clusters',facecolor = 'black',
            labelcolor = 'white')
        ax.set_title(np.unique(scdata.obs['sample'])[0])
    else:
        # ax.legend( loc = 'lower center')
        pass
        
    ax.autoscale(enable=True, tight=True)
    
   

In [8]:
def rotate(spatial_x,deg):
    """
    spatial_X: Nx2 matrix
    deg: rotation degree in radian
    
    """
    deg = np.deg2rad(deg)
    rotation_mat = np.array([[np.cos(deg),-1*np.sin(deg)],
                            [np.sin(deg),np.cos(deg)]])
    return np.matmul(spatial_x,rotation_mat)

In [21]:
sorted_sampleID

['H20.33.035.Cx26.MTG.02.007.1.01.03', 'H21.33.038.Cx20.MTG.02.007.3.01.02', 'H20.33.004.Cx26.MTG.02.007.1.01.05', 'H21.33.023.Cx26.MTG.02.007.1.03.05', 'H20.33.004.Cx26.MTG.02.007.1.02.03', ..., 'H21.33.028.Cx28.MTG.02.007.1.02.04', 'H21.33.005.Cx18.MTG.02.007.02.04', 'H21.33.006.Cx28.MTG.02.007.1.01.09.03', 'H21.33.021.Cx26.MTG.02.007.1.06', 'H21.33.006.Cx28.MTG.02.007.1.01.09.04']
Length: 69
Categories (69, object): ['H20.33.001.CX28.MTG.02.007.1.02.02', 'H20.33.001.CX28.MTG.02.007.1.02.03', 'H20.33.001.Cx28.MTG.02.007.1.01.03', 'H20.33.004.Cx26.MTG.02.007.1.01.04', ..., 'H21.33.038.Cx20.MTG.02.007.3.01.04', 'H21.33.040.Cx22.MTG.02.007.3.03.01', 'H21.33.040.Cx22.MTG.02.007.3.03.03', 'H21.33.040.Cx22.MTG.02.007.3.03.04']

In [ ]:
import numpy as np 
if True:
    adata.obs[['x','y']] = adata.obsm['X_spatial_raw'].copy()
    color_list = np.load(r'/mnt/Z/Lab/Bereket_public/Human_basal_ganglion_project/FigureMaking_notebooks/cluster_cmap_final_updated_9172025.npy',allow_pickle=True).item()
    color_list = list(color_list.values())
    color_map_dict = {nm:color_list[idx%len(color_list)] for idx,nm in enumerate(adata.obs['Subclass'].cat.categories)}
    point_size = {nm:0.3 for nm in adata.obs[f'Subclass'].unique() }
    all_clusters = np.unique(adata.obs[f'Subclass']).tolist()

plt.style.use('default')
fig,axs = plt.subplots(3,10,figsize=(30,10), dpi = 300)
sorted_sampleID = adata.obs['Section'].unique()
for idx,sampleID_ in enumerate(sorted_sampleID[30:61]):
    ax = axs[idx//10,idx%10]
    keep_cells =  (adata.obs['Section'] == sampleID_)
    plot_cluster_scdata(adata[keep_cells],color_map_dict,all_clusters,clusters=all_clusters,
                        bg_color = 'gray',
                        bg_pt_size = 0.055,
                        point_sz_map = point_size ,
                        transpose=1,flipx=1,flipy=1,hue_input = f'Subclass' ,
                        ax = ax,use_seaborn = True,show_title = False)
    ax.set_axis_off()
    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
_ = [ax.set_axis_off() for ax in axs.ravel()]
plt.show()